In [6]:
"""
AI Lab 04 + 05 -- ITERATIVE and SIMPLE version (no recursion)

Only 2 ideas are used in the whole file:
   BFS = a QUEUE  -> take from the FRONT  : queue.pop(0)
   DFS = a STACK  -> take from the BACK   : stack.pop()
Each item in the queue/stack is a whole PATH (a list of states),
so when we reach the goal we already have the answer.

Run:  python3 ai_lab_04_05_iterative.py
"""

# ---------------------------------------------------------------
# THE TEMPLATE (learn this once, every problem below is the same)
# ---------------------------------------------------------------
#   frontier = [ [start] ]            # list of paths
#   visited  = [start]
#   while frontier is not empty:
#       path  = take one path out     # pop(0) = BFS , pop() = DFS
#       state = last state of path
#       if state is the goal: return path
#       for each next_state of state:
#           if next_state not in visited:
#               visited.append(next_state)
#               frontier.append(path + [next_state])
#   return None                       # no solution


# ===============================================================
# 1. LAB 5 SAMPLE: BFS on a tree
# ===============================================================
tree = {'A': ['B', 'C'], 'B': ['D'], 'C': ['E', 'F'], 'D': ['G', 'H'],
        'E': ['I', 'J'], 'F': ['K', 'L'],
        'G': [], 'H': [], 'I': [], 'J': [], 'K': [], 'L': []}

def bfs_order(graph, start):
    queue = [start]
    order = []
    while len(queue) > 0:
        node = queue.pop(0)            # front
        order.append(node)
        for child in graph[node]:
            queue.append(child)        # back
    return order


In [7]:
print("1. BFS order      :", bfs_order(tree, 'A'))
    

1. BFS order      : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L']


In [3]:


# ===============================================================
# 2. LAB 5 SAMPLE: DFS on a graph WITH loops (iterative)
# ===============================================================
graph = {'A': ['B', 'C'], 'B': ['D', 'A'], 'C': ['E'], 'D': ['G', 'H', 'B'],
         'E': ['I', 'J'], 'F': ['K', 'L'], 'G': ['D'], 'H': ['D'],
         'I': ['E'], 'J': ['E'], 'K': ['F'], 'L': ['F']}

def dfs_order(graph, start, goal):
    stack = [start]
    visited = []
    while len(stack) > 0:
        node = stack.pop()             # back (most recent)
        if node in visited:
            continue                   # already done, skip it
        visited.append(node)
        if node == goal:
            return visited
        # push neighbours in REVERSE so the first neighbour is popped first
        neighbours = graph[node]
        for i in range(len(neighbours) - 1, -1, -1):
            if neighbours[i] not in visited:
                stack.append(neighbours[i])
    return visited



In [8]:
print("2. DFS order to J :", dfs_order(graph, 'A', 'J'))


2. DFS order to J : ['A', 'B', 'D', 'G', 'H', 'C', 'E', 'I', 'J']


In [9]:

# ===============================================================
# 3. EXERCISE 4.1: BFS on the weighted graph, S -> G
# ===============================================================
# each entry: node: [ (child, cost), ... ]   (alphabetical = tie break)
wgraph = {'S': [('A', 3), ('C', 2), ('D', 2)],
          'A': [],
          'B': [('E', 2)],
          'C': [('F', 1)],
          'D': [('B', 3), ('G', 8)],
          'E': [('G', 2)],
          'F': [('E', 0), ('G', 4)],
          'G': []}

def path_cost(path):
    total = 0
    for i in range(len(path) - 1):
        for child, cost in wgraph[path[i]]:
            if child == path[i + 1]:
                total = total + cost
    return total

def bfs_weighted(start, goal):
    frontier = [[start]]
    seen = [start]          # nodes already generated
    explored = []           # nodes already expanded (this is what the question asks)
    while len(frontier) > 0:
        path = frontier.pop(0)
        node = path[-1]
        explored.append(node)
        for child, cost in wgraph[node]:
            if child not in seen:
                seen.append(child)
                new_path = path + [child]
                if child == goal:                      # goal test when GENERATED
                    return explored, new_path, path_cost(new_path)
                frontier.append(new_path)
    return explored, None, None



In [10]:

    explored, path, cost = bfs_weighted('S', 'G')
    print("3. Ex4.1 explored :", explored)
    print("   path:", path, " cost:", cost)

3. Ex4.1 explored : ['S', 'A', 'C', 'D']
   path: ['S', 'D', 'G']  cost: 10


In [32]:

# ===============================================================
# 4. LAB 4: 8-PUZZLE  (state = string of 9 digits, '0' = blank)
# ===============================================================
START8 = "341256780"
GOAL8 = "123456780"

def puzzle_moves(state):
    """Return list of (action, new_state). Action = where the BLANK moves."""
    result = []
    b = state.index('0')               # position of blank, 0..8
    row = b // 3
    col = b % 3
    # (action name, row change, col change)
    for name, dr, dc in [('up', -1, 0), ('down', 1, 0), ('left', 0, -1), ('right', 0, 1)]:
        r = row + dr
        c = col + dc
        if 0 <= r < 3 and 0 <= c < 3:
            t = r * 3 + c              # tile position the blank swaps with
            letters = list(state)
            letters[b] = letters[t]
            letters[t] = '0'
            result.append((name, "".join(letters)))
    return result

def show_puzzle(state):
    print(state[0:3]); print(state[3:6]); print(state[6:9])

def puzzle_tree(start, max_depth):
    """Print the state space as a TREE, level by level (BFS, no loops back to parent)."""
    queue = [(start, '', 0)]           # (state, parent_state, depth)
    while len(queue) > 0:
        state, parent, depth = queue.pop(0)
        print("  " * depth + state)
        if depth < max_depth:
            for action, new_state in puzzle_moves(state):
                if new_state != parent:              # don't go straight back
                    queue.append((new_state, state, depth + 1))

def puzzle_bfs(start, goal):
    frontier = [(start, [])]           # (state, list of actions so far)
    visited = [start]
    while len(frontier) > 0:
        state, actions = frontier.pop(0)
        if state == goal:
            return actions
        for action, new_state in puzzle_moves(state):
            if new_state not in visited:
                visited.append(new_state)
                frontier.append((new_state, actions + [action]))
    return None


In [33]:
    print("4. 8-puzzle tree (depth 2):")

    answer = puzzle_bfs(START8, GOAL8)
    print("   solution:", len(answer), "moves:", answer)

4. 8-puzzle tree (depth 2):
   solution: 18 moves: ['up', 'up', 'left', 'down', 'right', 'up', 'left', 'left', 'down', 'right', 'right', 'up', 'left', 'left', 'down', 'right', 'right', 'down']


In [40]:


# ===============================================================
# 5. LAB 4 (Ex 3.4): FARMER, WOLF, CHICKEN, GRAIN
#    state = (farmer, wolf, chicken, grain)   0 = left, 1 = right
# ===============================================================
NAMES = ['farmer alone', 'wolf', 'chicken', 'grain']

def farmer_safe(state):
    farmer, wolf, chicken, grain = state
    if wolf == chicken and farmer != wolf:      # wolf eats chicken
        return False
    if chicken == grain and farmer != chicken:  # chicken eats grain
        return False
    return True

def farmer_next(state):
    result = []
    farmer = state[0]
    for who in range(4):               # 0 = alone, 1 = wolf, 2 = chicken, 3 = grain
        if who == 0 or state[who] == farmer:    # item must be on the farmer's side
            new = list(state)
            new[0] = 1 - farmer                 # farmer crosses
            if who != 0:
                new[who] = 1 - farmer           # item crosses too
            new = tuple(new)
            if farmer_safe(new):
                result.append((NAMES[who], new))
    return result

def farmer_bfs():
    start = (0, 0, 0, 0)
    goal = (1, 1, 1, 1)
    frontier = [(start, [])]
    visited = [start]
    while len(frontier) > 0:
        state, steps = frontier.pop(0)
        if state == goal:
            return steps
        for what, new in farmer_next(state):
            if new not in visited:
                visited.append(new)
                frontier.append((new, steps + [what + " -> " + str(new)]))
    return None



In [41]:
print("5. Farmer:")
for step in farmer_bfs():
    print("   take", step)


5. Farmer:
   take chicken -> (1, 0, 1, 0)
   take farmer alone -> (0, 0, 1, 0)
   take wolf -> (1, 1, 1, 0)
   take chicken -> (0, 1, 0, 0)
   take grain -> (1, 1, 0, 1)
   take farmer alone -> (0, 1, 0, 1)
   take chicken -> (1, 1, 1, 1)


In [27]:

# ===============================================================
# 6. EXERCISE 4.2: MISSIONARIES & CANNIBALS (BFS and DFS)
#    state = (missionaries_left, cannibals_left, boat)  boat: 1 = left, 0 = right
# ===============================================================
MC_START = (3, 3, 1)
MC_GOAL = (0, 0, 0)
BOAT_LOADS = [(1, 0), (0, 1), (1, 1), (2, 0), (0, 2)]   # (missionaries, cannibals)

def mc_ok(m, c):
    """Is it legal to have m missionaries and c cannibals on the LEFT bank?"""
    if m < 0 or m > 3 or c < 0 or c > 3:
        return False
    if m > 0 and c > m:                  # left bank: missionaries outnumbered
        return False
    mr = 3 - m                           # right bank
    cr = 3 - c
    if mr > 0 and cr > mr:               # right bank: missionaries outnumbered
        return False
    return True

def mc_next(state):
    m, c, boat = state
    result = []
    for dm, dc in BOAT_LOADS:
        if boat == 1:                    # boat goes left -> right
            new = (m - dm, c - dc, 0)
        else:                            # boat goes right -> left
            new = (m + dm, c + dc, 1)
        if mc_ok(new[0], new[1]):
            result.append(new)
    return result

def mc_search(use_bfs):
    frontier = [[MC_START]]
    visited = [MC_START]                 # repeated-state check
    while len(frontier) > 0:
        if use_bfs:
            path = frontier.pop(0)       # queue
        else:
            path = frontier.pop()        # stack
        state = path[-1]
        if state == MC_GOAL:
            return path
        for new in mc_next(state):
            if new not in visited:
                visited.append(new)
                frontier.append(path + [new])
    return None


In [29]:
print("6. Missionaries & Cannibals")
p = mc_search(True)
print("   BFS:", len(p) - 1, "crossings", p)
p = mc_search(False)
print("   DFS:", len(p) - 1, "crossings", p)

6. Missionaries & Cannibals
   BFS: 11 crossings [(3, 3, 1), (2, 2, 0), (3, 2, 1), (3, 0, 0), (3, 1, 1), (1, 1, 0), (2, 2, 1), (0, 2, 0), (0, 3, 1), (0, 1, 0), (1, 1, 1), (0, 0, 0)]
   DFS: 11 crossings [(3, 3, 1), (3, 1, 0), (3, 2, 1), (3, 0, 0), (3, 1, 1), (1, 1, 0), (2, 2, 1), (0, 2, 0), (0, 3, 1), (0, 1, 0), (0, 2, 1), (0, 0, 0)]


In [34]:

# ===============================================================
# 7. EXERCISE 4.3: LABYRINTH   (x = column, y = row  ->  lab[y][x])
# ===============================================================
labyrinth = [[0, 0, 0, 0, 0, 0, 1, 0],
             [0, 1, 0, 1, 1, 1, 1, 0],
             [0, 1, 1, 1, 0, 1, 0, 0],
             [0, 1, 0, 0, 0, 0, 0, 0],
             [0, 1, 1, 0, 1, 1, 3, 0],
             [0, 0, 1, 1, 1, 0, 0, 0],
             [0, 1, 2, 0, 1, 1, 1, 0],
             [0, 1, 0, 0, 0, 0, 0, 0]]

def draw(lab, visited=[]):
    """(a) ## wall, 2 spaces passage, D door, K key, '.' on visited cells."""
    for y in range(len(lab)):
        line = ""
        for x in range(len(lab[y])):
            v = lab[y][x]
            if v == 0:
                text = "##"
            elif v == 1:
                text = "  "
            elif v == 2:
                text = "D "
            else:
                text = "K "
            if (x, y) in visited:
                if v == 1:
                    text = ". "
                else:
                    text = text[0] + "."     # keep D or K, add the dot
            line = line + text
        print(line)

def neighbours(lab, x, y):
    """(b) adjacent cells (up, down, left, right) whose value is 1."""
    result = []
    height = len(lab)
    width = len(lab[0])
    if x < 0 or x >= width or y < 0 or y >= height:
        return result                         # input outside the maze
    for dx, dy in [(0, -1), (0, 1), (-1, 0), (1, 0)]:
        nx = x + dx
        ny = y + dy
        if 0 <= nx < width and 0 <= ny < height:     # stay inside the array
            if lab[ny][nx] == 1:
                result.append((nx, ny))
    return result

def solve(lab, start, goal):
    """(c) DFS with a stack. Each stack item is the path so far."""
    stack = [[start]]
    visited = [start]
    while len(stack) > 0:
        path = stack.pop()
        x, y = path[-1]
        if (x, y) == goal:
            return path
        for nb in neighbours(lab, x, y):
            if nb not in visited:
                visited.append(nb)
                stack.append(path + [nb])
    return None



In [35]:

# ===============================================================
# RUN EVERYTHING
# ===============================================================
if __name__ == "__main__":
    

    

   
    

    print("7. Labyrinth")
    draw(labyrinth)
    easy = []                                   # same maze but doors/keys count as passage
    for row in labyrinth:
        new_row = []
        for v in row:
            if v == 2 or v == 3:
                new_row.append(1)
            else:
                new_row.append(v)
        easy.append(new_row)
    print("   only 1s passable:", solve(labyrinth, (6, 0), (1, 7)))
    path = solve(easy, (6, 0), (1, 7))
    print("   doors open      :", path)
    draw(labyrinth, path)

7. Labyrinth
############  ##
##  ##        ##
##      ##  ####
##  ############
##    ##    K ##
####      ######
##  D ##      ##
##  ############
   only 1s passable: None
   doors open      : [(6, 0), (6, 1), (5, 1), (4, 1), (3, 1), (3, 2), (2, 2), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (2, 6), (1, 6), (1, 7)]
############. ##
##  ##. . . . ##
##. . . ##  ####
##. ############
##. . ##    K ##
####.     ######
##. D.##      ##
##. ############
